# Molecular simulation in Colab

Run the cells in order. CPU works for both examples. To try GPU4PySCF, select a GPU runtime and follow the optional GPU cell. TPU is not supported. The local MCP server does not remotely control this notebook. Runtime files are ephemeral; download results or copy them to Drive.

## Get the project
Publish your changes to the repository first, or upload/copy the project directory into `/content/colabexercise`. If cloning, set the repository URL and branch to the version containing this notebook.

In [ ]:
from pathlib import Path
import subprocess

project = Path('/content/colabexercise')
repo_url = 'https://github.com/gutama/colabexercise.git'
branch = 'main'  # Change to the published branch containing this project.
if not project.exists():
    subprocess.run(['git', 'clone', '--branch', branch, repo_url, str(project)], check=True)
if not (project / 'pyproject.toml').is_file():
    raise RuntimeError('Project files are missing; upload the project or select its published branch.')

In [ ]:
%pip install -e /content/colabexercise

## Optional NVIDIA GPU setup
Uncomment the installation below only if the runtime has a compatible CUDA 12 NVIDIA GPU. Consult the [GPU4PySCF installation guide](https://pyscf.org/user/gpu.html) for the current package versions and other CUDA versions. Restart the runtime if Colab requests it. GPU use can fail if the installed CUDA stack or calculation is unsupported; it is not silently replaced by CPU. This path has not been tested on a GPU in this project.

In [ ]:
# !nvidia-smi
# %pip install gpu4pyscf-cuda12x cutensor-cu12

## PySCF: H2 and integral export
Change the example JSON or the in-memory config for other systems. Set `device` to `gpu` after installing the GPU stack, or leave it at `cpu`.

In [ ]:
import os
import tempfile
os.environ['OMP_NUM_THREADS'] = '1'
from quantum_workflow.workflow import load_config, run

config = load_config(project / 'examples/h2-pyscf.json')
config.device = 'cpu'  # 'gpu' requires GPU4PySCF and a compatible GPU
pyscf_output = Path(tempfile.mkdtemp(prefix='pyscf-', dir='/content'))
summary = run(config, pyscf_output)
print(summary)
print('Output:', pyscf_output)

## Optional QE calculation (CPU)
Downloads micromamba and installs a separate conda-forge QE 7.6 CPU environment, plus an example H PBE pseudopotential. This takes several minutes and disk space. QE uses a periodic supercell; independently test vacuum and cutoffs. Review downloaded software and pseudopotential suitability and provenance before scientific use. This section does not export VQE-ready integrals.

In [ ]:
# Run this cell only if you want the QE example.
import tarfile
import shutil

mamba = Path('/content/micromamba')
qe_env = Path('/content/qe-env')
if not (qe_env / 'bin/pw.x').exists():
    subprocess.run(['curl', '-fL', 'https://micro.mamba.pm/api/micromamba/linux-64/latest', '-o', '/content/micromamba.tar.bz2'], check=True)
    with tarfile.open('/content/micromamba.tar.bz2', 'r:bz2') as tar:
        source = tar.extractfile('bin/micromamba')
        if source is None:
            raise RuntimeError('micromamba archive is missing its executable')
        with source, mamba.open('wb') as target:
            shutil.copyfileobj(source, target)
    mamba.chmod(0o755)
    subprocess.run([str(mamba), 'create', '-y', '-r', '/content/mamba-root', '-p', str(qe_env), '-c', 'conda-forge', 'qe=7.6'], check=True)
os.environ['PATH'] = str(qe_env / 'bin') + os.pathsep + os.environ['PATH']
pseudo_dir = project / 'pseudopotentials'
pseudo_dir.mkdir(exist_ok=True)
pseudo = pseudo_dir / 'H.pbe-kjpaw_psl.1.0.0.UPF'
subprocess.run(['curl', '-fL', 'https://pseudopotentials.quantum-espresso.org/upf_files/H.pbe-kjpaw_psl.1.0.0.UPF', '-o', str(pseudo)], check=True)
qe_config = load_config(project / 'examples/h2-qe.json')
qe_output = Path(tempfile.mkdtemp(prefix='qe-', dir='/content'))
print(run(qe_config, qe_output))
print('Output:', qe_output)

## Download PySCF results
The archive includes integrals and metadata. For QE, archive `qe_output` similarly; its scratch directory can be large. See the README for tensor conventions and an FCI consumption example.

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/pyscf-results', 'zip', pyscf_output)
files.download(archive)